In [1]:
import warnings
warnings.filterwarnings("ignore")

from datetime import date, datetime

import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from scipy.interpolate import griddata


In [7]:
TICKER     = "SPY"
MAX_TTM    =0.5     # years
ATM_BAND   = 0.10    # ±10% of spot

# ── Fetch spot ──────────────────────────────────────────────────────
tick = yf.Ticker(TICKER)
spot = tick.fast_info.get("lastPrice") or tick.fast_info.get("previousClose")
print(f"Spot = {spot:.2f}")

# ── Collect IV points ───────────────────────────────────────────────
strikes, ttms, ivs = [], [], []

for exp_str in tick.options:
    expiry = datetime.strptime(exp_str, "%Y-%m-%d").date()
    ttm = (expiry - date.today()).days / 365.0
    if ttm <= 0 or ttm > MAX_TTM:
        continue

    chain = tick.option_chain(exp_str)
    for df in (chain.calls, chain.puts):
        for row in df.itertuples(index=False):
            K  = float(row.strike)
            iv = float(row.impliedVolatility or 0)
            if iv <= 0 or K <= 0:
                continue
            if abs(K / spot - 1.0) > ATM_BAND:
                continue
            strikes.append(K)
            ttms.append(ttm)
            ivs.append(iv)

print(f"Collected {len(ivs)} IV points")

# ── Interpolate onto a regular grid ────────────────────────────────
strikes, ttms, ivs = np.array(strikes), np.array(ttms), np.array(ivs)

K_grid = np.linspace(strikes.min(), strikes.max(), 40)
T_grid = np.linspace(ttms.min(),   ttms.max(),   30)
KK, TT = np.meshgrid(K_grid, T_grid, indexing="ij")

iv_grid = griddata(
    np.column_stack([strikes, ttms]), ivs,
    (KK, TT), method="linear", fill_value=np.nan,
)
# fill remaining NaN with nearest
mask = np.isnan(iv_grid)
if mask.any():
    iv_grid[mask] = griddata(
        np.column_stack([strikes, ttms]), ivs,
        (KK[mask], TT[mask]), method="nearest",
    )

# ── Plot ────────────────────────────────────────────────────────────
fig = go.Figure(go.Surface(
    x=T_grid, y=K_grid, z=iv_grid,
    colorscale="Viridis",
    colorbar=dict(title="IV"),
))
fig.update_layout(
    title=f"{TICKER} — IV Surface (±{ATM_BAND:.0%} ATM, T ≤ {MAX_TTM}y)",
    scene=dict(xaxis_title="TTM (yrs)", yaxis_title="Strike", zaxis_title="IV"),
    width=900, height=650,
)
fig.show()


Spot = 689.07
Collected 3036 IV points


In [6]:

# ── IBKR version ────────────────────────────────────────────────────
# Requires TWS or IB Gateway running locally.
# Install: pip install ib_insync
# TWS paper: port 7497  |  TWS live: port 7496
# IB Gateway paper: port 4002  |  IB Gateway live: port 4001

from ib_insync import IB, Stock, Option

TICKER   = "SPY"
MAX_TTM  = 3.0
ATM_BAND = 0.10
HOST, PORT, CLIENT_ID = "127.0.0.1", 7497, 10
BATCH_SIZE = 50       # market-data lines per request (stay under your subscription limit)

ib = IB()
ib.connect(HOST, PORT, clientId=CLIENT_ID)

# ── Spot price ───────────────────────────────────────────────────────
stock = Stock(TICKER, "SMART", "USD")
ib.qualifyContracts(stock)
[und_ticker] = ib.reqTickers(stock)
spot = und_ticker.last or und_ticker.close or und_ticker.bid
print(f"Spot = {spot:.2f}")

# ── Option chain metadata ────────────────────────────────────────────
chains = ib.reqSecDefOptParams(stock.symbol, "", stock.secType, stock.conId)
chain  = next(c for c in chains if c.exchange == "SMART")

K_lo = spot * (1 - ATM_BAND)
K_hi = spot * (1 + ATM_BAND)

today = date.today()
contracts = []
for exp in sorted(chain.expirations):
    expiry = datetime.strptime(exp, "%Y%m%d").date()
    ttm = (expiry - today).days / 365.0
    if ttm <= 0 or ttm > MAX_TTM:
        continue
    for K in sorted(chain.strikes):
        if K_lo <= K <= K_hi:
            contracts.append(Option(TICKER, exp, K, "C", "SMART"))

ib.qualifyContracts(*contracts)
contracts = [c for c in contracts if c.conId]
print(f"Qualified {len(contracts)} contracts — requesting market data…")

# ── Fetch model greeks (IV) in batches ──────────────────────────────
all_tickers = []
for i in range(0, len(contracts), BATCH_SIZE):
    batch = contracts[i : i + BATCH_SIZE]
    tickers = ib.reqTickers(*batch)
    all_tickers.extend(tickers)
    ib.sleep(0.3)

ib.disconnect()

# ── Build scatter (K, TTM, IV) ───────────────────────────────────────
strikes_ib, ttms_ib, ivs_ib = [], [], []
for t in all_tickers:
    g = t.modelGreeks
    if g and g.impliedVol and g.impliedVol > 0:
        exp = t.contract.lastTradeDateOrContractMonth
        ttm = (datetime.strptime(exp, "%Y%m%d").date() - today).days / 365.0
        strikes_ib.append(t.contract.strike)
        ttms_ib.append(ttm)
        ivs_ib.append(g.impliedVol)

print(f"Collected {len(ivs_ib)} IV points from IBKR")

# ── Interpolate & plot ───────────────────────────────────────────────
strikes_ib = np.array(strikes_ib)
ttms_ib    = np.array(ttms_ib)
ivs_ib     = np.array(ivs_ib)

K_grid = np.linspace(strikes_ib.min(), strikes_ib.max(), 40)
T_grid = np.linspace(ttms_ib.min(),   ttms_ib.max(),   30)
KK, TT = np.meshgrid(K_grid, T_grid, indexing="ij")

iv_grid = griddata(np.column_stack([strikes_ib, ttms_ib]), ivs_ib,
                   (KK, TT), method="linear", fill_value=np.nan)
mask = np.isnan(iv_grid)
if mask.any():
    iv_grid[mask] = griddata(np.column_stack([strikes_ib, ttms_ib]), ivs_ib,
                              (KK[mask], TT[mask]), method="nearest")

fig = go.Figure(go.Surface(
    x=T_grid, y=K_grid, z=iv_grid,
    colorscale="Viridis",
    colorbar=dict(title="IV"),
))
fig.update_layout(
    title=f"{TICKER} — IV Surface via IBKR (±{ATM_BAND:.0%} ATM, T ≤ {MAX_TTM}y)",
    scene=dict(xaxis_title="TTM (yrs)", yaxis_title="Strike", zaxis_title="IV"),
    width=900, height=650,
)
fig.show()


RuntimeError: This event loop is already running

RuntimeError: This event loop is already running

API connection failed: ConnectionRefusedError(111, "Connect call failed ('127.0.0.1', 7497)")
Make sure API port on TWS/IBG is open
